In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel('merged_data.xlsx')
df["Donor"] = df["Donor"].astype("string").str.strip()
df["Treg"] = pd.to_numeric(df["Treg"].astype(
    "string").str.strip(), errors="coerce")

In [ ]:
# from src.flow.pipeline import summarize_categorical_data, plot_categorical_summary

# categorical_summary = summarize_categorical_data(df, include_missing=False)
# plot_categorical_summary(categorical_summary, variables=[
#                          "Sex", "Ethnicity", "Tissue"])

In [ ]:
import importlib
import src.flow.pipeline

importlib.reload(src.flow.pipeline)
src.flow.pipeline.plot_antibody_status_by_ethnicity(df)

In [ ]:
from src.flow.pipeline import analyze_tissue_demographics, analyze_tissue_demographic_group_combinations

df = df.rename(columns={
    "Ethnicity:": "Ethnicity",
    "Age (yr):": "Age",
    "BMI (kg/m^2):": "BMI",
    "Sex:": "Sex",
})

ethnicity_donor_counts = df.dropna(
    subset=["Donor", "Ethnicity"]).groupby("Ethnicity")["Donor"].nunique()
singleton_ethnicities = ethnicity_donor_counts[ethnicity_donor_counts == 1].index
df = df[~df["Ethnicity"].isin(singleton_ethnicities)].copy()
print(f"Removed singleton ethnicity groups: {list(singleton_ethnicities)}")

exclude_cols = {"Donor", "Ethnicity", "Age", "BMI", "Sex", "Tissue",
                "Ab CMV:", "Ab EBV (IgG):"}
subsets_to_test = [col for col in df.columns if col not in exclude_cols]

summary_stats_df = analyze_tissue_demographics(
    df, subset_columns=subsets_to_test
)
group_comparison_stats_df = analyze_tissue_demographic_group_combinations(
    df, subset_columns=subsets_to_test
)

In [ ]:
# cmv_by_donor = (
#     df.assign(
#         _CMV_status=df["Ab CMV:"].astype("string").str.strip().str.casefold()
#     )
#     .dropna(subset=["Donor"])
#     .groupby("Donor")["_CMV_status"]
#     .agg(lambda values: set(values.dropna()))
# )

# negative_donors = cmv_by_donor[
#     cmv_by_donor.map(lambda statuses: statuses == {"negative"})
# ].index

# df_cmv_negative = df[df["Donor"].isin(negative_donors)].drop(
#     columns=["Ab CMV:"]
# ).copy()

# print("CMV-negative donors:", df_cmv_negative["Donor"].nunique())
# display(
#     df_cmv_negative.groupby("Ethnicity")["Donor"].nunique()
#     .sort_values(ascending=False)
# )

# summary_stats_cmv_negative = src.flow.pipeline.analyze_tissue_demographics(
#     df_cmv_negative, subset_columns=subsets_to_test
# )
# group_stats_cmv_negative = (
#     src.flow.pipeline.analyze_tissue_demographic_group_combinations(
#         df_cmv_negative, subset_columns=subsets_to_test
#     )
# )

In [ ]:
significant_hits = summary_stats_df[summary_stats_df["Is_Significant_FDR"]]
significant_hits = significant_hits[~significant_hits["Demographic_Variable"].isin([
                                                                                   "Age", "BMI"])]
significant_hits[["Subset", "Tissue",
                  "Demographic_Variable", "Coefficient", "p_value", "p_value_FDR"]]

In [ ]:
from src.flow.pipeline import ols_summary
ols_summary(significant_hits)

In [ ]:
from src.flow.pipeline import plot_group_comparison_summary
plot_group_comparison_summary(group_comparison_stats_df)

In [ ]:
from src.flow.pipeline import visualize_ols_vs_mann_whitney
target_variable = "C(Ethnicity, Treatment(reference='Black or African American'))[T.Hispanic/Latino]"
target_q = summary_stats_df.loc[
    (summary_stats_df["Subset"] == "CD8 CD45RA-CCR7- CD69-CD103+") &
    (summary_stats_df["Tissue"] == "BLD") &
    (summary_stats_df["Demographic_Variable"]
     == "Ethnicity: Hispanic/Latino"),
    "p_value_FDR"
]
visualize_ols_vs_mann_whitney(
    df=df,
    population_name="CD8 CD45RA-CCR7- CD69-CD103+",
    tissue="BLD",
    target_variable=(
        "C(Ethnicity, Treatment(reference='Black or African American'))"
        "[T.Hispanic/Latino]"
    ),
    sex="MALE",
    group_comparison_stats_df=group_comparison_stats_df,
)

In [ ]:
import importlib
import src.flow.pipeline

importlib.reload(src.flow.pipeline)

population_name = "CD8 CD45RA+CCR7-"
target_variable = "C(Sex, Treatment(reference='MALE'))[T.FEMALE]"

src.flow.pipeline.visualize_ols_vs_mann_whitney(
    df=df,
    population_name=population_name,
    tissue="COL",
    target_variable=target_variable,
    ethnicity="Black or African American",
    group_comparison_stats_df=group_comparison_stats_df,
)

In [ ]:
from src.flow.pipeline import (
    summarize_donor_cohort,
    analyze_demographic_mixed_models,
)
import importlib
import src.flow.pipeline
importlib.reload(src.flow.pipeline)

cohort_audit = summarize_donor_cohort(df, subsets_to_test)
print(f"Unique donors: {cohort_audit['n_donors']}")
print(f"Donor-tissue observations: {cohort_audit['n_donor_tissue_rows']}")
display(cohort_audit['group_counts'])
display(cohort_audit['continuous_summary'])
display(cohort_audit['missingness'])

mixed_model_results, mixed_model_status, adjusted_group_means = analyze_demographic_mixed_models(
    df, subsets_to_test, min_donors=20, min_group_donors=5,
    return_marginal_means=True,
)
display(mixed_model_status['Status'].value_counts(
).rename_axis('Fit status').to_frame('Subsets'))
display(mixed_model_status[mixed_model_status['Status'] != 'fit'])
primary_demographic_hits = mixed_model_results[
    mixed_model_results['Demographic_Variable'].str.startswith(
        ('Sex:', 'Ethnicity:', 'Age', 'BMI')
    ) & mixed_model_results['Is_Significant_FDR']
].sort_values('p_value_FDR')
display(primary_demographic_hits.head(30))

In [ ]:
from src.flow.pipeline import (
    plot_demographic_mixed_model_findings,
    plot_adjusted_demographic_means,
    plot_demographic_effect_heatmap,
)
plot_demographic_mixed_model_findings(
    mixed_model_results,
    demographics=("Ethnicity", "Sex"),
    significant_only=True,
)
plot_adjusted_demographic_means(
    adjusted_group_means, mixed_model_results,
    demographics=("Ethnicity", "Sex"),
    significant_only=True,
)
plot_demographic_effect_heatmap(
    mixed_model_results, demographics=("Ethnicity", "Sex"),
    significant_only=False,
)

In [ ]:
# from src.flow.pipeline import plot_subset_frequency_by_tissue_ethnicity
# import importlib
# import src.flow.pipeline
# importlib.reload(src.flow.pipeline)

# ethnicity_tissue_fig, ethnicity_tissue_ax, ethnicity_tissue_counts = (
#     plot_subset_frequency_by_tissue_ethnicity(
#         df,
#         population_name="CD4 CD45RA+CCR7+",
#         min_donors=5,
#     )
# )
# display(ethnicity_tissue_counts)

In [ ]:
# from src.flow.pipeline import plot_subset_frequency_by_tissue_sex
# import importlib
# import src.flow.pipeline
# importlib.reload(src.flow.pipeline)

# sex_tissue_fig, sex_tissue_ax, sex_tissue_counts = (
#     plot_subset_frequency_by_tissue_sex(
#         df,
#         population_name="CD8 CD45RA-CCR7- CD69+CD103+",
#         min_donors=5,
#     )
# )
# display(sex_tissue_counts)

In [ ]:
import importlib
import src.flow.pipeline

importlib.reload(src.flow.pipeline)

ethnicity_tissue_interaction, ethnicity_tissue_interaction_status = (
    src.flow.pipeline.analyze_ethnicity_tissue_interaction(
        df,
        population_name="CD8 CD103+CD69+",
        focal_ethnicity="Hispanic/Latino",
        reference_ethnicity="White",
        min_donors_per_group=5,
    )
)

print(ethnicity_tissue_interaction_status)
display(ethnicity_tissue_interaction)
src.flow.pipeline.plot_ethnicity_tissue_adjusted_means(
    ethnicity_tissue_interaction, ethnicity_tissue_interaction_status
)

In [ ]:
from src.flow.pipeline import (
    analyze_sex_tissue_interaction,
    plot_sex_tissue_adjusted_means,
)
import importlib
import src.flow.pipeline
importlib.reload(src.flow.pipeline)

sex_tissue_interaction, sex_tissue_interaction_status = (
    analyze_sex_tissue_interaction(
        df,
        population_name="CD4 TEM CD69+CD103-",
        focal_sex="FEMALE",
        reference_sex="MALE",
        min_donors_per_group=5,
    )
)
print(sex_tissue_interaction_status)
display(sex_tissue_interaction)
plot_sex_tissue_adjusted_means(
    sex_tissue_interaction, sex_tissue_interaction_status
)